# Presence annotation
Scrub through a recording and mark the frames where a person **enters** and **leaves** the camera view.
Events are stored in `data/annotations/presence_events.csv` (`recording, frame_idx, event`), and a per-frame
`present` flag is derived from them (present from each ENTER until the next LEAVE; a recording that starts
with someone already in view: put an ENTER on frame 0).

Run all cells, then use the slider / Play / step buttons. **Save** writes the CSV (also done automatically on every change).

In [4]:
from io import BytesIO
from pathlib import Path
import numpy as np
import pandas as pd
import ipywidgets as w
from IPython.display import display
from datasets import load_from_disk
from PIL import Image, ImageDraw

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
ds = load_from_disk(str(ROOT / "data/baked/floor_posture"))
meta = ds.select_columns(["recording", "frame_idx"]).to_pandas()
# recordings are stored contiguously: dataset row = START[rec] + position within recording
START = meta.reset_index().groupby("recording")["index"].min().to_dict()
LEN = meta.groupby("recording").size().to_dict()
RECS = sorted(START)

def frame_png(rec, i, width=640):
    im = ds[START[rec] + i]["image"].convert("RGB")
    im = im.resize((width, int(im.height * width / im.width)))
    buf = BytesIO()
    im.save(buf, "PNG")
    return buf.getvalue()

In [7]:
rec_dd = w.Dropdown(options=RECS, description="Recording", layout=w.Layout(width="520px"))
slider = w.IntSlider(min=0, max=LEN[RECS[0]] - 1, value=0, continuous_update=False,
                     layout=w.Layout(width="720px"), description="Frame")
play = w.Play(min=0, max=LEN[RECS[0]] - 1, step=5, interval=150)
w.jslink((play, "value"), (slider, "value"))  # Play drives the slider in the browser
img = w.Image(format="png", width=640)
step_btns = [w.Button(description=d, layout=w.Layout(width="60px")) for d in ("-50", "-10", "-1", "+1", "+10", "+50")]
status = w.HTML()

OUT = ROOT / "data/annotations/presence_events.csv"  # tracked source annotations
events = pd.read_csv(OUT) if OUT.exists() else pd.DataFrame(columns=["recording", "frame_idx", "event"])
events = events.astype({"recording": str, "frame_idx": int, "event": str})

enter_btn = w.Button(description="ENTER ▶", button_style="success")
leave_btn = w.Button(description="LEAVE ◀", button_style="danger")
del_btn = w.Button(description="Delete event at this frame", button_style="warning")
ev_list = w.HTML()
strip = w.Image(format="png")

def rec_events(rec):
    return events[events.recording == rec].sort_values("frame_idx")

def present_at(rec, i):
    state = False
    for _, e in rec_events(rec).iterrows():
        if e.frame_idx <= i:
            state = e.event == "enter"
    return state

def timeline(rec, i, W=720, H=26):
    n = LEN[rec]
    bar = Image.new("RGB", (W, H), (200, 200, 200))
    d = ImageDraw.Draw(bar)
    ev = rec_events(rec)
    state, last = False, 0
    for _, e in ev.iterrows():
        x = int(e.frame_idx / n * W)
        if state:
            d.rectangle([last, 0, x, H], fill=(70, 170, 90))
        state, last = e.event == "enter", x
    if state:
        d.rectangle([last, 0, W, H], fill=(70, 170, 90))
    x = int(i / n * W)
    d.line([x, 0, x, H], fill=(0, 0, 0), width=3)
    buf = BytesIO(); bar.save(buf, "PNG")
    return buf.getvalue()

def save():
    events.sort_values(["recording", "frame_idx"]).to_csv(OUT, index=False)

def refresh(*_):
    rec, i = rec_dd.value, slider.value
    img.value = frame_png(rec, i)
    strip.value = timeline(rec, i)
    here = present_at(rec, i)
    status.value = (f"<b>{'PERSON PRESENT' if here else 'empty'}</b> &nbsp; frame {i}/{LEN[rec] - 1}")
    ev = rec_events(rec)
    ev_list.value = "<br>".join(f"{r.event.upper():5s} @ {r.frame_idx}" for r in ev.itertuples()) or "<i>no events</i>"

def add(kind):
    global events
    rec, i = rec_dd.value, slider.value
    events = events[~((events.recording == rec) & (events.frame_idx == i))]
    events = pd.concat([events, pd.DataFrame([{"recording": rec, "frame_idx": i, "event": kind}])], ignore_index=True)
    save(); refresh()

def delete(_):
    global events
    events = events[~((events.recording == rec_dd.value) & (events.frame_idx == slider.value))]
    save(); refresh()

def on_rec(_):
    n = LEN[rec_dd.value]
    slider.max = play.max = n - 1
    slider.value = 0
    refresh()

enter_btn.on_click(lambda _: add("enter"))
leave_btn.on_click(lambda _: add("leave"))
del_btn.on_click(delete)
for b in step_btns:
    b.on_click(lambda _, b=b: setattr(slider, "value", int(np.clip(slider.value + int(b.description), 0, slider.max))))
rec_dd.observe(on_rec, "value")
slider.observe(refresh, "value")
refresh()

display(w.VBox([
    rec_dd, status, img, strip, slider,
    w.HBox([play] + step_btns),
    w.HBox([enter_btn, leave_btn, del_btn]),
    ev_list,
]))

In [8]:
# Per-frame presence table derived from the events (all recordings); handy for training / checking
rows = []
for rec in RECS:
    ev = rec_events(rec)
    present = np.zeros(LEN[rec], bool)
    state, last = False, 0
    for r in ev.itertuples():
        present[last:r.frame_idx] = state
        state, last = r.event == "enter", r.frame_idx
    present[last:] = state
    rows.append(pd.DataFrame({"recording": rec, "frame_idx": np.arange(LEN[rec]), "present": present}))
frames = pd.concat(rows, ignore_index=True)
annotated = events.recording.nunique()
print(f"{annotated}/{len(RECS)} recordings annotated; {frames.present.mean():.1%} of frames present")
frames.to_csv(ROOT / "data/baked/presence_frames.csv", index=False)

22/25 recordings annotated; 85.6% of frames present
